# Multi-Source Visual Search: MSSearch V2 Benchmark (Google Colab Pro)

Evaluates our full MSSearch V2 visual search engine on `200_samples.json` under candidate pool `N_20` (default).
Employs SAM3, SigLIP, continuous enclosure sub-canvases, and clean foveated reading (`<image>\n<question>`).
Outputs are atomically appended per sample to JSONL in Google Drive to support uninterrupted auto-resumption.

In [ ]:
# Cell 1: Mount Google Drive & Environment Configuration
import os
import sys
from google.colab import drive

# EDIT THIS PATH: Root directory in your Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/cvpr_research"
drive.mount("/content/drive")

DATASET_PATH = os.path.join(DRIVE_ROOT, "dataset", "200_samples.json")
IMAGE_DIR = os.path.join(DRIVE_ROOT, "dataset", "images")
MODELS_DIR = os.path.join(DRIVE_ROOT, "models")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "results_our")

SAM3_PATH = os.path.join(MODELS_DIR, "sam3", "sam3.pt")
SIGLIP_NAME = os.path.join(MODELS_DIR, "siglip-base-patch16-384")
NLP_MODEL = os.path.join(MODELS_DIR, "spacy", "en_core_web_sm-3.8.0")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("Loaded HF_TOKEN from Colab Secrets.")
except Exception:
    print("No HF_TOKEN secret found; proceeding with public access.")

print(f"Dataset Path : {DATASET_PATH}")
print(f"Image Dir    : {IMAGE_DIR}")
print(f"Output Dir   : {OUTPUT_DIR}")

In [ ]:
# Cell 2: Install Dependencies
!pip install -q transformers accelerate torchvision pillow sentencepiece spacy scikit-learn
!python -m spacy download en_core_web_sm

In [ ]:
# Cell 3: Clone MSSearch Repository (or Set Local Path)
REPO_DIR = "/content/MSSearch"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/your-org/MSSearch.git {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    sys.path.insert(0, os.path.join(REPO_DIR, "experiments", "our"))
print("Repository path configured.")

In [ ]:
# Cell 4: Launch MSSearch V2 Evaluation
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"  # Choose any from our 8 benchmark models
POOL_MODE = "N_20"                         # Default candidate pool size
CONFIG_FILE = f"{REPO_DIR}/experiments/our/config_default.json"

!python {REPO_DIR}/experiments/our/eval_runner.py \
    --model_path "{MODEL_NAME}" \
    --sam3_path "{SAM3_PATH}" \
    --siglip_name "{SIGLIP_NAME}" \
    --nlp_model "{NLP_MODEL}" \
    --dataset_path "{DATASET_PATH}" \
    --image_dir "{IMAGE_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --config "{CONFIG_FILE}" \
    --pool_mode "{POOL_MODE}" \
    --resume